# V3：扩展特征准备

复用 V3 CLIN 的队列、结局和固定划分，只处理开发集，不读取留出集标签。

时间轨迹：入 ICU 后0–24小时，化验同次 hadm_id、生命体征同次 stay_id；取首尾差值、每小时斜率、SD、后12小时减前12小时均值、测量次数。相同时刻先平均。少于两个时刻时斜率和差值缺失，缺失不删除患者。

ICD 文档：只纳入索引住院开始前已结束的住院；同时保留 ICD9/10 版本、确切码与三级类目。没有既往诊断的患者仍保留，文档为空。此步只准备文档，没有提前拟合 TF-IDF/SVD。

V3 静态列仍沿用原来的−6至24小时窗口，因此轨迹增加的是动态信息而非重定义原始基线。描述性缺失率只用于了解数据，后续缺失率筛选在训练折内进行。

主分析继承 V3 的20%变量缺失规则及8个白细胞分类变量豁免。轨迹变量没有豁免，测量次数是检查强度指标，不应作因果风险因素解读。

**balanced 简化计算版**：固定基模型参数、full保留5×10外层验证；LASSO内部5折/30λ，stacking内部3折。与上级严格版使用独立缓存；前6折旧预测不混入。当前没有运行新版本真实患者模型。

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# 从扩展目录或 notebooks 子目录定位项目，避免硬编码 Windows 盘符。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dn_extensions" / "settings.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请从 dn_mortality_repro_V3_extensions 或其 notebooks 子目录启动。")
sys.path.insert(0, str(ROOT))
print("扩展目录：", ROOT)


In [ ]:
# 默认只展示说明，不扫描数据库。准备特征时改为 True。
RUN_FEATURE_PREPARATION = False
if RUN_FEATURE_PREPARATION:
    from dn_extensions.features import build_features
    from dn_extensions.reporting import update_preparation_notebook
    manifest = build_features()
    update_preparation_notebook(manifest)
else:
    print("balanced共享上级已准备的特征，无需重新prepare；缺少特征时到上级运行prepare。")


## 实验状态与结论（自动更新）

简化版尚未运行真实患者实验。原严格版前6折已保存，未混入此版。